In [1]:
import pandas as pd
import os
from geopy.geocoders import Nominatim
import pandas as pd
import time
import numpy as np

In [2]:
# Caminho que você está tentando usar
caminho = "/home/felipe/Projeto/Portfolio/Portfolio2/Regression_PriceHouse/src//data/raw/imoveis_todos_completo.csv"

# Verifique se o arquivo existe
print("O arquivo existe?", os.path.exists(caminho))
print("É um arquivo?", os.path.isfile(caminho) if os.path.exists(caminho) else "Arquivo não encontrado")

O arquivo existe? True
É um arquivo? True


In [3]:
df = pd.read_csv(caminho)

df

,preco,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta
0,R$ 2.300.000,Rua João Silvio de Lara Machado,"Sobrado para comprar com 354 m², 4 quartos, 5 ...",4.0,5.0,354.0,https://www.zapimoveis.com.br/imovel/venda-sob...,ponta-grossa,pr,2026-02-09 08:28:42
1,R$ 2.994.975,Rua Visconde de Baraúna,Lote/Terreno para comprar com 3524 m² emJardim...,NaN,NaN,3524.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42
2,R$ 310.000,Rua Jaguapitã,Lote/Terreno para comprar com 40 m² emBoa Vist...,NaN,NaN,40.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42
3,R$ 350.000,Rua Luiz Migdalski,"Casa de condomínio para comprar com 165 m², 4 ...",4.0,3.0,165.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 08:28:42
4,R$ 202.696,Rua Eduardo Burgardt,Lote/Terreno para comprar com 200 m² emContorn...,NaN,NaN,200.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42
...,...,...,...,...,...,...,...,...,...,...
8596,R$ 1.310.000,Rua Lauro Marcondes Ferreira,"Casa para comprar com 310 m², 1 quarto, 1 banh...",1.0,1.0,310.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 09:09:06
8597,R$ 350.000,Rua Vereador Ernâni Batista Rosas,Lote/Terreno para comprar com 10 m² emJardim C...,NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06
8598,R$ 2.700.000,Rodovia BR-373,"Lote/Terreno para comprar com 10 m² emChapada,...",NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06
8599,R$ 1.200.000,Rua Benjamin Constant,"Apartamento para comprar com 123 m², 3 quartos...",3.0,3.0,123.0,https://www.zapimoveis.com.br/imovel/venda-apa...,ponta-grossa,pr,2026-02-09 09:09:06


In [4]:
df["vagas_garagem"] = (
    df["endereco"]
    .str.extract(r"(\d+)\s+vagas?", expand=False)
    .astype(float)
)

df["bairro"] = (
    df["endereco"]
    .str.extract(r"em\s*([^,]+)", expand=False)
    .str.replace(r"^Imóvel\s+", "", regex=True)
    .str.strip()
)

df["tipo_imovel"] = (
    df["endereco"]
    .str.extract(r"^(.*?)\s+para comprar", expand=False)
    .str.strip()
)

df["preco"] = (
    df["preco"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)


df

,preco,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel
0,2300000.0,Rua João Silvio de Lara Machado,"Sobrado para comprar com 354 m², 4 quartos, 5 ...",4.0,5.0,354.0,https://www.zapimoveis.com.br/imovel/venda-sob...,ponta-grossa,pr,2026-02-09 08:28:42,2.0,Jardim Carvalho,Sobrado
1,2994975.0,Rua Visconde de Baraúna,Lote/Terreno para comprar com 3524 m² emJardim...,NaN,NaN,3524.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Jardim Carvalho,Lote/Terreno
2,310000.0,Rua Jaguapitã,Lote/Terreno para comprar com 40 m² emBoa Vist...,NaN,NaN,40.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Boa Vista,Lote/Terreno
3,350000.0,Rua Luiz Migdalski,"Casa de condomínio para comprar com 165 m², 4 ...",4.0,3.0,165.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Estrela,Casa de condomínio
4,202696.0,Rua Eduardo Burgardt,Lote/Terreno para comprar com 200 m² emContorn...,NaN,NaN,200.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Contorno,Lote/Terreno
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8596,1310000.0,Rua Lauro Marcondes Ferreira,"Casa para comprar com 310 m², 1 quarto, 1 banh...",1.0,1.0,310.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Casa
8597,350000.0,Rua Vereador Ernâni Batista Rosas,Lote/Terreno para comprar com 10 m² emJardim C...,NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Lote/Terreno
8598,2700000.0,Rodovia BR-373,"Lote/Terreno para comprar com 10 m² emChapada,...",NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Chapada,Lote/Terreno
8599,1200000.0,Rua Benjamin Constant,"Apartamento para comprar com 123 m², 3 quartos...",3.0,3.0,123.0,https://www.zapimoveis.com.br/imovel/venda-apa...,ponta-grossa,pr,2026-02-09 09:09:06,2.0,Centro,Apartamento


In [5]:
df["preco_m2"] = df["preco"] / df["area_m2"]

df

,preco,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2
0,2300000.0,Rua João Silvio de Lara Machado,"Sobrado para comprar com 354 m², 4 quartos, 5 ...",4.0,5.0,354.0,https://www.zapimoveis.com.br/imovel/venda-sob...,ponta-grossa,pr,2026-02-09 08:28:42,2.0,Jardim Carvalho,Sobrado,6497.175141
1,2994975.0,Rua Visconde de Baraúna,Lote/Terreno para comprar com 3524 m² emJardim...,NaN,NaN,3524.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Jardim Carvalho,Lote/Terreno,849.879398
2,310000.0,Rua Jaguapitã,Lote/Terreno para comprar com 40 m² emBoa Vist...,NaN,NaN,40.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Boa Vista,Lote/Terreno,7750.000000
3,350000.0,Rua Luiz Migdalski,"Casa de condomínio para comprar com 165 m², 4 ...",4.0,3.0,165.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Estrela,Casa de condomínio,2121.212121
4,202696.0,Rua Eduardo Burgardt,Lote/Terreno para comprar com 200 m² emContorn...,NaN,NaN,200.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Contorno,Lote/Terreno,1013.480000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8596,1310000.0,Rua Lauro Marcondes Ferreira,"Casa para comprar com 310 m², 1 quarto, 1 banh...",1.0,1.0,310.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Casa,4225.806452
8597,350000.0,Rua Vereador Ernâni Batista Rosas,Lote/Terreno para comprar com 10 m² emJardim C...,NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Lote/Terreno,35000.000000
8598,2700000.0,Rodovia BR-373,"Lote/Terreno para comprar com 10 m² emChapada,...",NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Chapada,Lote/Terreno,270000.000000
8599,1200000.0,Rua Benjamin Constant,"Apartamento para comprar com 123 m², 3 quartos...",3.0,3.0,123.0,https://www.zapimoveis.com.br/imovel/venda-apa...,ponta-grossa,pr,2026-02-09 09:09:06,2.0,Centro,Apartamento,9756.097561


In [6]:
df["endereco_geo"] = (
    df["rua"].str.strip() + ", " +
    df["bairro"].str.strip() + ", Ponta Grossa, PR, Brasil"
)


In [7]:
#Verificar os endereços unicos
enderecos_unicos = df["endereco_geo"].dropna().unique()
len(enderecos_unicos)

1325

In [8]:
%%time

geolocator = Nominatim(user_agent="imoveis-ml")

#def geocode_endereco(bairro):
#    try:
#        location = geolocator.geocode(
#            f"{bairro}, Ponta Grossa, PR, Brasil",
#            timeout=10
#        )
#        time.sleep(1)  # obrigatório para Nominatim
#        if location:
#            return location.latitude, location.longitude
#    except:
#        return None, None

#df[["lat", "lon"]] = df["bairro"].apply(
#    lambda x: pd.Series(geocode_endereco(x))
#)

cache = {}

for i, endereco in enumerate(enderecos_unicos, start=1):
    if endereco in cache:
        continue

    try:
        location = geolocator.geocode(endereco, timeout=10)
        time.sleep(1)  # obrigatório (Nominatim)

        if location:
            cache[endereco] = (location.latitude, location.longitude)
        else:
            cache[endereco] = (None, None)

    except Exception as e:
        cache[endereco] = (None, None)

    if i % 50 == 0:
        print(f"{i}/{len(enderecos_unicos)} endereços processados")


50/1325 endereços processados
100/1325 endereços processados
150/1325 endereços processados
200/1325 endereços processados
250/1325 endereços processados
300/1325 endereços processados
350/1325 endereços processados
400/1325 endereços processados
450/1325 endereços processados
500/1325 endereços processados
550/1325 endereços processados
600/1325 endereços processados
650/1325 endereços processados
700/1325 endereços processados
750/1325 endereços processados
800/1325 endereços processados
850/1325 endereços processados
900/1325 endereços processados
950/1325 endereços processados
1000/1325 endereços processados
1050/1325 endereços processados
1100/1325 endereços processados
1150/1325 endereços processados
1200/1325 endereços processados
1250/1325 endereços processados
1300/1325 endereços processados
CPU times: user 2.74 s, sys: 385 ms, total: 3.12 s
Wall time: 35min 59s


In [9]:
#salvar o cache
cache_df = (
    pd.DataFrame.from_dict(
        cache,
        orient="index",
        columns=["lat", "lon"]
    )
    .reset_index()
    .rename(columns={"index": "endereco_geo"})
)

cache_df.to_csv("cache_geocoding.csv", index=False)


In [10]:
#unir o conjunto de dados com o cache
df = df.merge(
    cache_df,
    on="endereco_geo",
    how="left"
)

df

,preco,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2,endereco_geo,lat,lon
0,2300000.0,Rua João Silvio de Lara Machado,"Sobrado para comprar com 354 m², 4 quartos, 5 ...",4.0,5.0,354.0,https://www.zapimoveis.com.br/imovel/venda-sob...,ponta-grossa,pr,2026-02-09 08:28:42,2.0,Jardim Carvalho,Sobrado,6497.175141,"Rua João Silvio de Lara Machado, Jardim Carval...",NaN,NaN
1,2994975.0,Rua Visconde de Baraúna,Lote/Terreno para comprar com 3524 m² emJardim...,NaN,NaN,3524.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Jardim Carvalho,Lote/Terreno,849.879398,"Rua Visconde de Baraúna, Jardim Carvalho, Pont...",NaN,NaN
2,310000.0,Rua Jaguapitã,Lote/Terreno para comprar com 40 m² emBoa Vist...,NaN,NaN,40.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Boa Vista,Lote/Terreno,7750.000000,"Rua Jaguapitã, Boa Vista, Ponta Grossa, PR, Br...",-25.067383,-50.177596
3,350000.0,Rua Luiz Migdalski,"Casa de condomínio para comprar com 165 m², 4 ...",4.0,3.0,165.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Estrela,Casa de condomínio,2121.212121,"Rua Luiz Migdalski, Estrela, Ponta Grossa, PR,...",-25.107402,-50.174850
4,202696.0,Rua Eduardo Burgardt,Lote/Terreno para comprar com 200 m² emContorn...,NaN,NaN,200.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Contorno,Lote/Terreno,1013.480000,"Rua Eduardo Burgardt, Contorno, Ponta Grossa, ...",-25.136835,-50.206803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8596,1310000.0,Rua Lauro Marcondes Ferreira,"Casa para comprar com 310 m², 1 quarto, 1 banh...",1.0,1.0,310.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Casa,4225.806452,"Rua Lauro Marcondes Ferreira, Jardim Carvalho,...",-25.079336,-50.153754
8597,350000.0,Rua Vereador Ernâni Batista Rosas,Lote/Terreno para comprar com 10 m² emJardim C...,NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Lote/Terreno,35000.000000,"Rua Vereador Ernâni Batista Rosas, Jardim Carv...",NaN,NaN
8598,2700000.0,Rodovia BR-373,"Lote/Terreno para comprar com 10 m² emChapada,...",NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Chapada,Lote/Terreno,270000.000000,"Rodovia BR-373, Chapada, Ponta Grossa, PR, Brasil",-24.935198,-50.922139
8599,1200000.0,Rua Benjamin Constant,"Apartamento para comprar com 123 m², 3 quartos...",3.0,3.0,123.0,https://www.zapimoveis.com.br/imovel/venda-apa...,ponta-grossa,pr,2026-02-09 09:09:06,2.0,Centro,Apartamento,9756.097561,"Rua Benjamin Constant, Centro, Ponta Grossa, P...",-25.094068,-50.155871


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8601 entries, 0 to 8600
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   preco          8601 non-null   float64
 1   rua            7958 non-null   object 
 2   endereco       8601 non-null   object 
 3   quartos        6488 non-null   float64
 4   banheiros      6727 non-null   float64
 5   area_m2        8591 non-null   float64
 6   link           8601 non-null   object 
 7   cidade         8601 non-null   object 
 8   estado         8601 non-null   object 
 9   data_coleta    8601 non-null   object 
 10  vagas_garagem  6011 non-null   float64
 11  bairro         8591 non-null   object 
 12  tipo_imovel    8500 non-null   object 
 13  preco_m2       8591 non-null   float64
 14  endereco_geo   7949 non-null   object 
 15  lat            6988 non-null   float64
 16  lon            6988 non-null   float64
dtypes: float64(8), object(9)
memory usage: 1.1+ MB


## 1️⃣ Identificar quem falhou

In [12]:
mask_falha = df["lat"].isna() | df["lon"].isna()

df_falha = df.loc[mask_falha, ["rua", "bairro"]].drop_duplicates()
df_falha.shape


(238, 2)

## 2️⃣ Tentar novamente: Rua + Cidade (sem bairro)

In [13]:
df_falha["endereco_alt"] = (
    df_falha["rua"].str.strip() +
    ", Ponta Grossa, PR, Brasil"
)

cache_alt = {}

for endereco in df_falha["endereco_alt"].unique():
    try:
        location = geolocator.geocode(endereco, timeout=3)
        time.sleep(1)

        if location:
            cache_alt[endereco] = (location.latitude, location.longitude)
        else:
            cache_alt[endereco] = (None, None)

    except:
        cache_alt[endereco] = (None, None)


## 3️⃣ Aplicar o resultado no dataframe principal

In [14]:
cache_alt_df = (
    pd.DataFrame.from_dict(
        cache_alt,
        orient="index",
        columns=["lat_alt", "lon_alt"]
    )
    .reset_index()
    .rename(columns={"index": "endereco_alt"})
)

df = df.merge(
    cache_alt_df,
    left_on=(
        df["rua"].str.strip() + ", Ponta Grossa, PR, Brasil"
    ),
    right_on="endereco_alt",
    how="left"
)

df["lat"] = df["lat"].fillna(df["lat_alt"])
df["lon"] = df["lon"].fillna(df["lon_alt"])

df.drop(columns=["lat_alt", "lon_alt", "endereco_alt"], inplace=True)


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8601 entries, 0 to 8600
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   preco          8601 non-null   float64
 1   rua            7958 non-null   object 
 2   endereco       8601 non-null   object 
 3   quartos        6488 non-null   float64
 4   banheiros      6727 non-null   float64
 5   area_m2        8591 non-null   float64
 6   link           8601 non-null   object 
 7   cidade         8601 non-null   object 
 8   estado         8601 non-null   object 
 9   data_coleta    8601 non-null   object 
 10  vagas_garagem  6011 non-null   float64
 11  bairro         8591 non-null   object 
 12  tipo_imovel    8500 non-null   object 
 13  preco_m2       8591 non-null   float64
 14  endereco_geo   7949 non-null   object 
 15  lat            8131 non-null   float64
 16  lon            8131 non-null   float64
dtypes: float64(8), object(9)
memory usage: 1.1+ MB


## 4️⃣ Fallback final: Bairro + Cidade (último recurso)

In [16]:
mask_final = df["lat"].isna()

df.loc[mask_final, "endereco_geo"] = (
    df.loc[mask_final, "bairro"] +
    ", Ponta Grossa, PR, Brasil"
)


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8601 entries, 0 to 8600
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   preco          8601 non-null   float64
 1   rua            7958 non-null   object 
 2   endereco       8601 non-null   object 
 3   quartos        6488 non-null   float64
 4   banheiros      6727 non-null   float64
 5   area_m2        8591 non-null   float64
 6   link           8601 non-null   object 
 7   cidade         8601 non-null   object 
 8   estado         8601 non-null   object 
 9   data_coleta    8601 non-null   object 
 10  vagas_garagem  6011 non-null   float64
 11  bairro         8591 non-null   object 
 12  tipo_imovel    8500 non-null   object 
 13  preco_m2       8591 non-null   float64
 14  endereco_geo   7949 non-null   object 
 15  lat            8131 non-null   float64
 16  lon            8131 non-null   float64
dtypes: float64(8), object(9)
memory usage: 1.1+ MB


## 📊 Métrica de qualidade (recomendo salvar)

In [18]:
df["nivel_geocoding"] = np.select(
    [
        df["lat"].notna() & df["rua"].notna() & df["bairro"].notna(),
        df["lat"].notna() & df["rua"].notna(),
        df["lat"].notna() & df["bairro"].notna()
    ],
    [
        "rua_bairro",
        "rua",
        "bairro"
    ],
    default="falhou"
)

df

,preco,rua,endereco,quartos,banheiros,area_m2,link,cidade,estado,data_coleta,vagas_garagem,bairro,tipo_imovel,preco_m2,endereco_geo,lat,lon,nivel_geocoding
0,2300000.0,Rua João Silvio de Lara Machado,"Sobrado para comprar com 354 m², 4 quartos, 5 ...",4.0,5.0,354.0,https://www.zapimoveis.com.br/imovel/venda-sob...,ponta-grossa,pr,2026-02-09 08:28:42,2.0,Jardim Carvalho,Sobrado,6497.175141,"Rua João Silvio de Lara Machado, Jardim Carval...",-25.022443,-50.120978,rua_bairro
1,2994975.0,Rua Visconde de Baraúna,Lote/Terreno para comprar com 3524 m² emJardim...,NaN,NaN,3524.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Jardim Carvalho,Lote/Terreno,849.879398,"Jardim Carvalho, Ponta Grossa, PR, Brasil",NaN,NaN,falhou
2,310000.0,Rua Jaguapitã,Lote/Terreno para comprar com 40 m² emBoa Vist...,NaN,NaN,40.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Boa Vista,Lote/Terreno,7750.000000,"Rua Jaguapitã, Boa Vista, Ponta Grossa, PR, Br...",-25.067383,-50.177596,rua_bairro
3,350000.0,Rua Luiz Migdalski,"Casa de condomínio para comprar com 165 m², 4 ...",4.0,3.0,165.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Estrela,Casa de condomínio,2121.212121,"Rua Luiz Migdalski, Estrela, Ponta Grossa, PR,...",-25.107402,-50.174850,rua_bairro
4,202696.0,Rua Eduardo Burgardt,Lote/Terreno para comprar com 200 m² emContorn...,NaN,NaN,200.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 08:28:42,NaN,Contorno,Lote/Terreno,1013.480000,"Rua Eduardo Burgardt, Contorno, Ponta Grossa, ...",-25.136835,-50.206803,rua_bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8596,1310000.0,Rua Lauro Marcondes Ferreira,"Casa para comprar com 310 m², 1 quarto, 1 banh...",1.0,1.0,310.0,https://www.zapimoveis.com.br/imovel/venda-cas...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Casa,4225.806452,"Rua Lauro Marcondes Ferreira, Jardim Carvalho,...",-25.079336,-50.153754,rua_bairro
8597,350000.0,Rua Vereador Ernâni Batista Rosas,Lote/Terreno para comprar com 10 m² emJardim C...,NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Jardim Carvalho,Lote/Terreno,35000.000000,"Jardim Carvalho, Ponta Grossa, PR, Brasil",NaN,NaN,falhou
8598,2700000.0,Rodovia BR-373,"Lote/Terreno para comprar com 10 m² emChapada,...",NaN,NaN,10.0,https://www.zapimoveis.com.br/imovel/venda-ter...,ponta-grossa,pr,2026-02-09 09:09:06,NaN,Chapada,Lote/Terreno,270000.000000,"Rodovia BR-373, Chapada, Ponta Grossa, PR, Brasil",-24.935198,-50.922139,rua_bairro
8599,1200000.0,Rua Benjamin Constant,"Apartamento para comprar com 123 m², 3 quartos...",3.0,3.0,123.0,https://www.zapimoveis.com.br/imovel/venda-apa...,ponta-grossa,pr,2026-02-09 09:09:06,2.0,Centro,Apartamento,9756.097561,"Rua Benjamin Constant, Centro, Ponta Grossa, P...",-25.094068,-50.155871,rua_bairro


In [19]:
df["nivel_geocoding"].value_counts(normalize=True) * 100

nivel_geocoding
rua_bairro    87.013138
bairro         7.464248
falhou         5.476107
rua            0.046506
Name: proportion, dtype: float64

In [20]:
# Caminho que você está tentando usar
caminho2 = "/home/felipe/Projeto/Portfolio/Portfolio2/Regression_PriceHouse/data/pre/preprocessed.csv"

df.to_csv(caminho2, index=False)

## Importar e salvar os conjuntos de dados referente a feature engineering. 

In [3]:
import osmnx as ox

cidade = "Guarapuava, Paraná, Brasil"

#-----------------------------------------------------
tags_hospitais = {"amenity": "hospital"}

hospitais = ox.features_from_place(cidade, tags_hospitais)
#----------------------------------------------------------

mercados = ox.features_from_place(
    cidade,
    {"shop": ["supermarket", "convenience"]}
)

#---------------------------------------------------------
farmacias = ox.features_from_place(
    cidade,
    {"amenity": "pharmacy"}
)

parques = ox.features_from_place(
    cidade,
    tags={
        "leisure": ["park", "garden"],
        "landuse": "recreation_ground"
    }
)

In [4]:
parques

geometry  \
element id                                                               
node    6379778069                         POINT (-51.45833 -25.39156)   
        12618670330                        POINT (-51.40081 -25.39298)   
way     295514840    POLYGON ((-51.47107 -25.39661, -51.47102 -25.3...   
        355334031    POLYGON ((-51.46775 -25.38274, -51.46776 -25.3...   
        355338223    POLYGON ((-51.47317 -25.37665, -51.46958 -25.3...   
...                                                                ...   
        1152326596   POLYGON ((-51.48715 -25.3831, -51.4868 -25.383...   
        1311396688   POLYGON ((-51.48839 -25.38155, -51.48853 -25.3...   
        1311396689   POLYGON ((-51.47034 -25.38886, -51.47055 -25.3...   
        1387902153   POLYGON ((-51.48404 -25.39787, -51.4843 -25.39...   
        1459569413   POLYGON ((-51.4833 -25.39775, -51.48325 -25.39...   

                    leisure                                      name  \
element id                                                              
node    6379778069   garden  Canteiro central da Avenida Manoel Ribas   
        12618670330    park                     Hope Valley Adventure   
way     295514840      park                            Parque do Lago   
        355334031      park                                       NaN   
        355338223      park                       Parque das Crianças   
...                     ...                                       ...   
        1152326596     park                                       NaN   
        1311396688     park                Parque Simeão Varela de Sà   
        1311396689     park                           Praça da escola   
        1387902153     park                         Praça Vista Batel   
        1459569413   garden                                       NaN   

                        access  dog                            operator  \
element id                                                                
node    6379778069         NaN  NaN                                 NaN   
        12618670330  customers  NaN                                 NaN   
way     295514840          NaN  NaN                                 NaN   
        355334031          NaN  NaN                                 NaN   
        355338223          NaN  yes  Prefeitura Municipal de Guarapuava   
...                        ...  ...                                 ...   
        1152326596         NaN  NaN                                 NaN   
        1311396688     private  NaN                                 NaN   
        1311396689         NaN  NaN                                 NaN   
        1387902153         NaN  NaN                         Vista Batel   
        1459569413         NaN  NaN                                 NaN   

                      addr:city             addr:street addr:suburb  \
element id                                                            
node    6379778069          NaN                     NaN         NaN   
        12618670330         NaN                     NaN         NaN   
way     295514840           NaN                     NaN         NaN   
        355334031           NaN                     NaN         NaN   
        355338223           NaN                     NaN         NaN   
...                         ...                     ...         ...   
        1152326596          NaN                     NaN         NaN   
        1311396688   Guarapuava  Rua Simão Varela de Sá  Vila Carli   
        1311396689          NaN                     NaN         NaN   
        1387902153          NaN            Rua Gerânios       Batel   
        1459569413          NaN                     NaN         NaN   

                    addr:country amenity landuse addr:postcode description  \
element id                                                                   
node    6379778069           NaN     NaN     NaN           NaN         NaN   
        1261867

In [5]:
escolas = ox.features_from_place(
    cidade,
    tags={
        "amenity": ["school", "college", "university"]
    }
)
escolas

geometry  \
element id                                                              
node    5300956064                        POINT (-51.48877 -25.38355)   
        5322637456                        POINT (-51.48741 -25.55633)   
        5322637457                        POINT (-51.48851 -25.55704)   
        5371452226                        POINT (-51.46325 -25.39028)   
        5371452227                        POINT (-51.46412 -25.39113)   
        6379948080                         POINT (-51.4572 -25.38507)   
        6401371097                        POINT (-51.47463 -25.41334)   
        6401371100                        POINT (-51.46957 -25.41848)   
way     403693895   POLYGON ((-51.46297 -25.40752, -51.46362 -25.4...   
        405782164   POLYGON ((-51.48927 -25.39534, -51.48867 -25.3...   
        406892874   POLYGON ((-51.47027 -25.40568, -51.46944 -25.4...   
        429332605   POLYGON ((-51.47215 -25.40351, -51.47164 -25.4...   
        429332961   POLYGON ((-51.47035 -25.40399, -51.47085 -25.4...   
        429333832   POLYGON ((-51.4965 -25.38659, -51.49617 -25.38...   
        429481816   POLYGON ((-51.48118 -25.3505, -51.47848 -25.34...   
        446409043   POLYGON ((-51.4625 -25.42475, -51.46249 -25.42...   
        482331377   POLYGON ((-51.49669 -25.39468, -51.4973 -25.39...   
        482333609   POLYGON ((-51.4876 -25.39458, -51.4867 -25.394...   
        486892335   POLYGON ((-51.43341 -25.35568, -51.43339 -25.3...   
        556623428   POLYGON ((-51.46378 -25.39013, -51.46337 -25.3...   
        556637320   POLYGON ((-51.46487 -25.3911, -51.4643 -25.390...   
        556655595   POLYGON ((-51.48949 -25.38497, -51.48911 -25.3...   
        556655596   POLYGON ((-51.48948 -25.38491, -51.48912 -25.3...   
        557492411   POLYGON ((-51.48947 -25.38447, -51.48915 -25.3...   
        557495346   POLYGON ((-51.4466 -25.38437, -51.44621 -25.38...   
        571041604   POLYGON ((-51.49617 -25.40927, -51.496 -25.408...   
        601839703   POLYGON ((-51.4656 -25.39072, -51.46569 -25.39...   
        610110803   POLYGON ((-51.48529 -25.39389, -51.48526 -25.3...   
        622736026   POLYGON ((-51.46169 -25.37055, -51.46228 -25.3...   
        637902163   POLYGON ((-51.28162 -25.3752, -51.28192 -25.37...   
        660473357   POLYGON ((-51.47938 -25.37228, -51.47952 -25.3...   
        675419275   POLYGON ((-51.47082 -25.39174, -51.47129 -25.3...   
        676839017   POLYGON ((-51.45268 -25.40312, -51.45273 -25.4...   
        681289802   POLYGON ((-51.47167 -25.35057, -51.47061 -25.3...   
        681291295   POLYGON ((-51.44238 -25.39264, -51.44213 -25.3...   
        681291840   POLYGON ((-51.44886 -25.39421, -51.44914 -25.3...   
        1081263800  POLYGON ((-51.49301 -25.40084, -51.49295 -25.4...   
        1081268413  POLYGON ((-51.49075 -25.40399, -51.49054 -25.4...   
        1163457677  POLYGON ((-51.52072 -25.39586, -51.52019 -25.3...   
        1163457678  POLYGON ((-51.5201 -25.39583, -51.52021 -25.39...   

                       amenity  \
element id                       
node    5300956064  university   
        5322637456      school   
        5322637457      school   
        5371452226  university   
        5371452227      school   
        6379948080  university   
        6401371097      school   
        6401371100      school   
way     403693895       school   
        405782164       school   
        406892874   university   
        429332605      college   
        429332961      college   
        429333832   university   
        429481816   university   
        446409043   university   
        482331377       school   
        482333609      college   
        486892335       school   
        556623428   university   
        556637320       school   
        556655595   university   
        556655596   university   
        557492411   university   
        557495346       school   
        571041604       school   
        601839703       school   
        

In [6]:
escolas["geometry_std"] = escolas.geometry.apply(
    lambda g: g.centroid if g.geom_type != "Point" else g
)

escolas_std = escolas.set_geometry("geometry_std")
escolas_std.geometry.geom_type.value_counts()


Point    40
Name: count, dtype: int64

In [7]:
import re
import pandas as pd

def classificar_escola(row):
    nome = str(row.get("name", "")).lower()

    padroes_publicos = [
        "municipal",
        "estadual",
        "federal",
        "instituto federal",
        "colégio estadual",
        "escola municipal",
        "cmei", "emei", "emef", "eef", "eeef",
        "universidade federal",
        "universidade estadual",
        "ifpr", "ifsp", "ifsc", "ifrs"
    ]

    for p in padroes_publicos:
        if p in nome:
            return "publica"

    return "privada"

escolas_std["tipo_escola"] = escolas_std.apply(classificar_escola, axis=1)


In [8]:
escolas_std["tipo_escola"].value_counts(normalize=True)

tipo_escola
privada    0.55
publica    0.45
Name: proportion, dtype: float64

In [9]:
policia = ox.features_from_place(
    cidade,
    tags={
        "amenity": ["police", "fire_station", "courthouse"]
    }
)
policia

geometry  \
element id                                                              
node    1844728255                        POINT (-51.51812 -25.28509)   
way     429481830   POLYGON ((-51.47091 -25.35286, -51.47053 -25.3...   
        538558746   POLYGON ((-51.26127 -25.37153, -51.26117 -25.3...   
        821807734   POLYGON ((-51.46126 -25.3956, -51.46054 -25.39...   

                         amenity                          name  \
element id                                                       
node    1844728255        police    Polícia Militar Rodoviária   
way     429481830   fire_station  Corpo de Bombeiros Primavera   
        538558746         police    Polícia Rodoviária Federal   
        821807734         police               Polícia Federal   

                            police building opening_hours  \
element id                                                  
node    1844728255  traffic_police      NaN           NaN   
way     429481830              NaN      NaN           NaN   
        538558746   traffic_police      yes          24/7   
        821807734              NaN      NaN           NaN   

                                      operator operator:wikidata  \
element id                                                         
node    1844728255                         NaN               NaN   
way     429481830                          NaN               NaN   
        538558746   Polícia Rodoviária Federal          Q1158853   
        821807734    Polícia Federal do Brasil          Q2289697   

                               operator:wikipedia             phone source  \
element id                                                                   
node    1844728255                            NaN               NaN    NaN   
way     429481830                             NaN               NaN    NaN   
        538558746   pt:Polícia Rodoviária Federal  +55 41 3535 2132   Bing   
        821807734    pt:Polícia Federal do Brasil  +55 42 3035 8719    NaN   

                     addr:city addr:housenumber addr:postcode  \
element id                                                      
node    1844728255         NaN              NaN           NaN   
way     429481830          NaN              NaN           NaN   
        538558746          NaN              NaN           NaN   
        821807734   Guarapuava             1950     85010-170   

                             addr:street addr:suburb                 email  \
element id                                                                   
node    1844728255                   NaN         NaN                   NaN   
way     429481830                    NaN         NaN                   NaN   
        538558746                    NaN         NaN                   NaN   
        821807734   Rua Professor Becker      Centro  gab.gpb.pr@pf.gov.br   

                                 operator:en  
element id                                    
node    1844728255                       NaN  
way     429481830                        NaN  
        538558746                        NaN  
        821807734   Federal Police of Brazil

In [38]:
farmacias

geometry  \
element id                                                              
node    1031943563                        POINT (-50.15588 -25.09352)   
        1031943860                        POINT (-50.16401 -25.09035)   
        1031943863                        POINT (-50.16147 -25.09029)   
        1770873384                        POINT (-50.15568 -25.07788)   
        1770963734                        POINT (-50.16653 -25.07928)   
        3192179958                         POINT (-50.16062 -25.0931)   
        7161634469                        POINT (-50.15635 -25.05468)   
        7516035689                        POINT (-50.14237 -25.17824)   
way     563456150   POLYGON ((-50.10402 -25.09705, -50.10389 -25.0...   

                     amenity dispensing                           name  \
element id                                                               
node    1031943563  pharmacy        yes                Farmacia Nissei   
        1031943860  pharmacy        yes                            NaN   
        1031943863  pharmacy        yes             Farmacia Erva Doce   
        1770873384  pharmacy        NaN               Farmácia Fleming   
        1770963734  pharmacy        NaN                            NaN   
        3192179958  pharmacy        NaN               Farmácia Fleming   
        7161634469  pharmacy        yes                        Fleming   
        7516035689  pharmacy        yes             Farmácia Bom Jesus   
way     563456150   pharmacy        NaN  Laboratório Farmacêutico UEPG   

                              source healthcare     addr:city  \
element id                                                      
node    1031943563  Bing;Tracksource        NaN           NaN   
        1031943860  Bing;Tracksource        NaN           NaN   
        1031943863  Bing;Tracksource        NaN           NaN   
        1770873384               NaN   pharmacy           NaN   
        1770963734               NaN        NaN           NaN   
        3192179958               NaN        NaN  Ponta Grossa   
        7161634469               NaN   pharmacy           NaN   
        7516035689               NaN   pharmacy  Ponta Grossa   
way     563456150                NaN   pharmacy           NaN   

                   addr:housenumber                addr:street addr:suburb  \
element id                                                                   
node    1031943563              NaN                        NaN         NaN   
        1031943860              NaN                        NaN         NaN   
        1031943863              NaN                        NaN         NaN   
        1770873384              NaN                        NaN         NaN   
        1770963734              NaN                        NaN         NaN   
        3192179958              284  Praça Barão do Rio Branco      Centro   
        7161634469              NaN                        NaN         NaN   
        7516035689              NaN                 Rua 14 Bis   Cará-Cará   
way     563456150               NaN                        NaN         NaN   

                   opening_hours building operator  
element id                                          
node    1031943563           NaN      NaN      NaN  
        1031943860           NaN      NaN      NaN  
        1031943863           NaN      NaN      NaN  
        1770873384           NaN      NaN      NaN  
        1770963734           NaN      NaN      NaN  
        3192179958          24/7      NaN      NaN  
        7161634469           NaN      NaN      NaN  
        7516035689           NaN      NaN      NaN  
way     563456150            NaN      yes     UEPG

In [39]:
mercados.head(50)

geometry  \
element id                                                               
node    848166969                          POINT (-50.17768 -25.08725)   
        849022448                          POINT (-50.16098 -25.10613)   
        849022572                           POINT (-50.15904 -25.1059)   
        1031118407                         POINT (-50.17928 -25.10504)   
        1031118993                         POINT (-50.18012 -25.10728)   
        1770956552                          POINT (-50.15976 -25.0884)   
        1770965034                         POINT (-50.16689 -25.08012)   
        3669193081                          POINT (-50.14995 -25.0726)   
        4146475840                           POINT (-50.124 -25.09305)   
        4259656190                         POINT (-50.15562 -25.05714)   
        4259716689                         POINT (-50.15586 -25.05718)   
        4268870994                          POINT (-50.1557 -25.11073)   
        4268874596                         POINT (-50.15771 -25.05661)   
        4268881289                          POINT (-50.1617 -25.05695)   
        4337761893                         POINT (-50.17035 -25.06002)   
        4481040592                         POINT (-50.17863 -25.10495)   
        5578333911                         POINT (-49.82668 -25.13761)   
        6700628469                          POINT (-50.2242 -25.05471)   
        7516035690                         POINT (-50.14313 -25.17785)   
        7516035691                         POINT (-50.14002 -25.17651)   
        11486104802                        POINT (-50.12456 -25.12296)   
        11486115937                        POINT (-50.12442 -25.12423)   
        11860913772                        POINT (-49.96099 -25.14291)   
way     70948753     POLYGON ((-50.17134 -25.09146, -50.17117 -25.0...   
        71255400     POLYGON ((-50.14525 -25.09052, -50.1446 -25.09...   
        88825551     POLYGON ((-50.15386 -25.07348, -50.15345 -25.0...   
        88899512     POLYGON ((-50.14389 -25.09077, -50.14284 -25.0...   
        90184903     POLYGON ((-50.15879 -25.12552, -50.15794 -25.1...   
        165752796    POLYGON ((-50.15757 -25.09396, -50.15715 -25.0...   
        255570109    POLYGON ((-50.16594 -25.08046, -50.16612 -25.0...   
        256241593    POLYGON ((-50.16584 -25.08049, -50.16579 -25.0...   
        256246337    POLYGON ((-50.16067 -25.07553, -50.16058 -25.0...   
        256346881    POLYGON ((-50.17976 -25.0838, -50.17902 -25.08...   
        312054750    POLYGON ((-50.19975 -25.10238, -50.19988 -25.1...   
        470435303    POLYGON ((-50.11483 -25.09517, -50.11462 -25.0...   
        470435306    POLYGON ((-50.11461 -25.09364, -50.11324 -25.0...   
        470435310    POLYGON ((-50.11309 -25.09105, -50.11232 -25.0...   
        548042719    POLYGON ((-50.15866 -25.10557, -50.15879 -25.1...   
        562461636    POLYGON ((-50.12545 -25.10345, -50.12545 -25.1...   
        1178006932   POLYGON ((-50.20872 -25.09186, -50.20858 -25.0...   

                                                    name         shop  \
element id                                                              
node    848166969                  Mercado Santo Antonio  supermarket   
        849022448                   Supermercado Tozzeto  supermarket   
        849022572                   Supermercado Muffato  supermarket   
        1031118407                      Mercado Vlastuin  supermarket   
        1031118993                       Mercado Dmenjon  supermarket   
        1770956552                                   NaN  convenience   
        1770965034                                   NaN  convenience   
        3669193081   Condor Super Center Jardim Carvalho  supermarket   
        4146475840                            Super Gype  supermarket   
        4259656190                    Mercearia Maranata  supermarket   
        4259716689                       Mercado Atlanta  supermarket   
    

In [28]:
hospitais

geometry  \
element id                                                               
node    4259722589                         POINT (-50.16337 -25.05867)   
        4287184389                         POINT (-50.15929 -25.10113)   
        12256464715                        POINT (-50.20163 -25.10218)   
        12256491257                        POINT (-50.15895 -25.10153)   
way     71255444     POLYGON ((-50.16555 -25.10429, -50.16541 -25.1...   
        165530986    POLYGON ((-50.17679 -25.08611, -50.17657 -25.0...   
        311477265    POLYGON ((-50.12751 -25.10039, -50.12751 -25.1...   
        311478181    POLYGON ((-50.1621 -25.08877, -50.16198 -25.08...   
        311481151    POLYGON ((-50.13647 -25.05584, -50.13642 -25.0...   
        565081124    POLYGON ((-50.12691 -25.10074, -50.12667 -25.1...   
        1074765245   POLYGON ((-50.17073 -25.0885, -50.17095 -25.08...   
        1076771218   POLYGON ((-50.10491 -25.09863, -50.10495 -25.0...   
        1324503646   POLYGON ((-50.2016 -25.10199, -50.20166 -25.10...   
        1324506468   POLYGON ((-50.15924 -25.10101, -50.15974 -25.1...   
        1324741610   POLYGON ((-50.15421 -25.12632, -50.15422 -25.1...   

                                            addr:street   amenity  \
element id                                                          
node    4259722589               Rua Nilza Marques Neme  hospital   
        4287184389                    Rua Augusto Ribas  hospital   
        12256464715                                 NaN  hospital   
        12256491257                                 NaN  hospital   
way     71255444     Rua Doutor Joaquim de Paula Xavier  hospital   
        165530986                  Avenida Dom Pedro II  hospital   
        311477265                  Rua Dolarico Correia  hospital   
        311478181           Rua Doutor Francisco Búrzio  hospital   
        311481151               Avenida Monteiro Lobato  hospital   
        565081124                                   NaN  hospital   
        1074765245                   Rua Pastor Fugmann  hospital   
        1076771218             Alameda Nabuco de Araújo  hospital   
        1324503646                                  NaN  hospital   
        1324506468                                  NaN  hospital   
        1324741610                                  NaN  hospital   

                                                                  name  \
element id                                                               
node    4259722589                                  CAS posto de saúde   
        4287184389                                      Pronto Socorro   
        12256464715                                    UPA Santa Paula   
        12256491257                     HU Hospital Universitário UEPG   
way     71255444           Hospital da Criança João Vargas de Oliveira   
        165530986                                   Hospital Bom Jesus   
        311477265                        Hospital Vicentino São Camilo   
        311478181           Santa Casa de Misericórdia de Ponta Grossa   
        311481151                                  Hospital São Camilo   
        565081124                                            Vicentino   
        1074765245                                 Hospital Evangélico   
        1076771218   Hospital Universitário Regional dos Campos Gerais   
        1324503646                                                 NaN   
        1324506468                                                 NaN   
        1324741610                                                 NaN   

                    addr:housenumber opening_hours            phone  \
element id                                                            
node    4259722589               NaN           NaN              NaN   
        4287184389                81          24/7    +554232207800   
        12256464715              NaN           NaN              NaN   
        1225649

In [47]:
escolas_std.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/escolas.csv",
    index=False
)

hospitais.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/hospitais.csv",
    index=False
)

parques.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/parques.csv",
    index=False
)

mercados.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/mercados.csv",
    index=False
)

farmacias.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/farmacia.csv",
    index=False
)

policia.to_csv(
    "/home/felipe/Projeto/Portfolio/Portfolio2/"
    "Regression_PriceHouse/data/pre/policia.csv",
    index=False
)